It's better to test on the converge of the model.

In [1]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [16]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [1]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [18]:
import random
from keras.optimizers import SGD

In [19]:
from sklearn.datasets import make_classification

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

In [20]:
train_pool = 53440
test_size = 500
train_sizes=[53440]
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]

In [21]:
df = pd.read_csv("diamonds.csv")

In [22]:
median_price = df["price"].median()
df["label"] = (df["price"] > median_price).astype(int)
df = df.drop(columns=['price'])

In [23]:
df['id'] = np.arange(1, len(df) + 1)
print(df)

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0                60.6            60.0        6

In [24]:
print(df)
print(df["label"].value_counts())

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0                60.6            60.0        6

In [25]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [26]:
print(df_train_pool.head())
print(df_test.head())

   features/carat  features/clarity  features/color  features/cut  \
0            1.26                 2               4             2   
1            0.80                 3               4             4   
2            0.56                 4               2             4   
3            1.51                 3               6             1   
4            0.33                 6               5             4   

   features/depth  features/table  features/x  features/y  features/z  label  \
0            60.6            60.0        6.97        7.00        4.23      1   
1            62.1            54.0        5.96        5.99        3.71      1   
2            61.7            54.0        5.28        5.32        3.27      0   
3            64.0            58.0        7.24        7.27        4.64      1   
4            62.2            54.0        4.43        4.45        2.76      0   

   id  
0   1  
1   2  
2   3  
3   4  
4   5  
   features/carat  features/clarity  features/color  fea

In [27]:
nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]

In [28]:
train_df = nested_train_dfs[0]

In [29]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[1.2600e+00 2.0000e+00 4.0000e+00 ... 7.0000e+00 4.2300e+00 1.0000e-10]
 [8.0000e-01 3.0000e+00 4.0000e+00 ... 5.9900e+00 3.7100e+00 2.0000e-10]
 [5.6000e-01 4.0000e+00 2.0000e+00 ... 5.3200e+00 3.2700e+00 3.0000e-10]
 ...
 [3.5000e-01 3.0000e+00 2.0000e+00 ... 4.5200e+00 2.7900e+00 5.3438e-06]
 [5.6000e-01 4.0000e+00 3.0000e+00 ... 5.3000e+00 3.2800e+00 5.3439e-06]
 [1.2500e+00 3.0000e+00 2.0000e+00 ... 6.8900e+00 4.2500e+00 5.3440e-06]]


In [30]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[5.4000e-01 5.0000e+00 3.0000e+00 ... 5.3100e+00 3.2500e+00 5.3441e-06]
 [5.0000e-01 5.0000e+00 3.0000e+00 ... 5.0900e+00 3.1600e+00 5.3442e-06]
 [4.1000e-01 5.0000e+00 5.0000e+00 ... 4.8000e+00 2.9500e+00 5.3443e-06]
 ...
 [3.0000e-01 4.0000e+00 5.0000e+00 ... 4.3200e+00 2.6900e+00 5.3938e-06]
 [3.6000e-01 3.0000e+00 2.0000e+00 ... 4.5700e+00 2.8200e+00 5.3939e-06]
 [7.0000e-01 1.0000e+00 2.0000e+00 ... 5.7700e+00 3.4900e+00 5.3940e-06]]


In [43]:
test_df["label"].value_counts()

0    259
1    241
Name: label, dtype: int64

In [31]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

2026-06-30 23:19:09.329085: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38477 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:31:00.0, compute capability: 8.0


# Fold 1

In [32]:
from tensorflow.keras.regularizers import l2

In [33]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 150
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

2026-06-30 23:19:10.975303: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:630] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


209/209 - 2s - loss: 0.4180 - accuracy: 0.8271 - val_loss: 0.2461 - val_accuracy: 0.9220 - 2s/epoch - 8ms/step
209/209 - 0s - loss: 0.1712 - accuracy: 0.9549 - val_loss: 0.1741 - val_accuracy: 0.9300 - 284ms/epoch - 1ms/step
209/209 - 0s - loss: 0.1165 - accuracy: 0.9670 - val_loss: 0.1952 - val_accuracy: 0.9040 - 284ms/epoch - 1ms/step
209/209 - 0s - loss: 0.0986 - accuracy: 0.9686 - val_loss: 0.2609 - val_accuracy: 0.8640 - 279ms/epoch - 1ms/step
209/209 - 0s - loss: 0.0884 - accuracy: 0.9700 - val_loss: 0.1432 - val_accuracy: 0.9340 - 286ms/epoch - 1ms/step
209/209 - 0s - loss: 0.0822 - accuracy: 0.9705 - val_loss: 0.1449 - val_accuracy: 0.9340 - 278ms/epoch - 1ms/step
209/209 - 0s - loss: 0.0792 - accuracy: 0.9704 - val_loss: 0.1538 - val_accuracy: 0.9340 - 283ms/epoch - 1ms/step
209/209 - 0s - loss: 0.0785 - accuracy: 0.9704 - val_loss: 0.0673 - val_accuracy: 0.9720 - 282ms/epoch - 1ms/step
209/209 - 0s - loss: 0.0762 - accuracy: 0.9710 - val_loss: 0.0668 - val_accuracy: 0.9760 - 

In [34]:
train_loss_values = unreduced_loss_fn(y_train, model.predict(X_train)).numpy()

1670/1670 [==============================] - 1s 596us/step


In [35]:
from tqdm import tqdm

IF

In [36]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [37]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(64))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(64), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in tqdm(enumerate(explanation_ds.as_numpy_iterator()),total=num_test_samples,desc="Computing influence"):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
2026-06-30 23:20:07.417803: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x2a553e60
Computing influence: 100%|██████████| 500/500 [03:50<00:00,  2.17it/s]


       Train_ID         Score
0             1  5.897499e-07
1             2  3.293020e-04
2             3  9.945452e-04
3             4  2.390611e-08
4             5  7.273074e-07
...         ...           ...
53435     53436  1.148820e-06
53436     53437  5.799646e-07
53437     53438  2.994647e-06
53438     53439  6.687135e-04
53439     53440  2.509319e-08

[53440 rows x 2 columns]


TC

In [38]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(64), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in tqdm(enumerate(explanation_ds.as_numpy_iterator()),total=num_test_samples,desc="Computing influence"):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

Computing influence: 100%|██████████| 500/500 [05:30<00:00,  1.51it/s]


       Train_ID         Score
0             1 -7.071174e-12
1             2 -6.287536e-08
2             3  2.581977e-06
3             4 -2.756835e-13
4             5  3.725036e-10
...         ...           ...
53435     53436  5.466266e-10
53436     53437  3.160546e-10
53437     53438  1.370666e-09
53438     53439  5.188303e-07
53439     53440 -2.925853e-13

[53440 rows x 2 columns]


To have more direct view of that, try to match the ranking of two dataframe.

In [39]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [40]:
TracIn_sorted.to_csv("Diamonds_TC_Train_Set_1.csv",index = False)
df_sorted.to_csv("Diamonds_IF_Train_Set_1.csv",index = False)

In [41]:
df_train_loss = pd.DataFrame({
    'Train_ID': train_ids,
    'Loss': train_loss_values
}).sort_values('Loss', ascending=False).reset_index(drop=True)
print(df_train_loss)

       Train_ID       Loss
0         52314  57.831642
1         33840  31.985361
2         33644  31.140001
3         40576  29.950096
4         12222  29.715176
...         ...        ...
53435     16323   0.000000
53436     31412   0.000000
53437     48740   0.000000
53438     16333   0.000000
53439     20670   0.000000

[53440 rows x 2 columns]


In [42]:
df_train_loss.to_csv("Diamonds_Train_Loss_Train_Set_1.csv",index = False)